Load the dataset

In [1]:
import pandas as pd
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")

In [2]:
import pandas as pd
outp = pd.read_parquet("output/judgeRAG/rag_outputs.parquet")
outp

,query_id,query,answer,retrieved_ids,latency_sec
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,Kort: in de verstrekte zoekresultaten is geen ...,"[27587, 2549, 27675, 14752, 20443, 13673, 9355...",52.305740
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort antwoord: in principe oefenen de ouders h...,"[3790, 3791, 3794, 3812, 3797, 3811, 3796, 393...",17.306959
2,305,Moet ik branddetectors installeren in Brussel?,Ja. In het Brussels Gewest moet elke woning mi...,"[21608, 23173, 6930, 2247, 14630, 1783, 1704, ...",17.577047
3,1811,"Als ik mijn consumentenkrediet niet betaal, ku...",Kort: ja — in bepaalde omstandigheden kunnen (...,"[9505, 4856, 14614, 15632, 14533, 26606, 15646...",52.234624
4,1558,Wat te doen als er een nieuwe huisgenoot in Br...,Kort en praktisch: geef de komst van de nieuwe...,"[4124, 4108, 26606, 152, 26740, 25728, 26739, ...",59.051880
...,...,...,...,...,...
1430,1106,De bal van mijn kind is in de tuin van mijn bu...,Kort antwoord: ja — u mag uw bal laten terugge...,"[2715, 3780, 2782, 27419, 2710, 26462, 4278, 2...",30.579444
1431,134,Hoe lang moet men wachten voordat men een besl...,U ontvangt in principe binnen 30 dagen na ontv...,"[26678, 26705, 26666, 26709, 26708, 26676, 266...",19.322466
1432,1031,Ik woon samen met mijn partner en/of mijn kind...,Kort antwoord: bij een verbreking van de samen...,"[3816, 106, 25835, 3791, 25833, 1074, 3724, 36...",24.015115
1433,1715,Welke documenten en informatie mag het OCMW ei...,Kort gezegd mag het OCMW (het centrum) alle in...,"[26674, 26744, 26676, 27880, 26673, 1574, 2569...",82.663040


In [3]:
#pip install deepeval

In [4]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from tqdm import tqdm

goldens = [Golden(input=question) for question in meta_qa.question]
dataset = EvaluationDataset(goldens=goldens)

for i, golden in enumerate(tqdm(dataset.goldens, desc="Building test cases")):
    test_case = LLMTestCase(
        input=meta_qa.question[i],
        actual_output=outp.answer[i],
        expected_output=meta_qa.answer[i]
    )
    dataset.add_test_case(test_case)

Building test cases: 100%|██████████| 1435/1435 [00:00<00:00, 5683.51it/s]


In [5]:
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from deepeval.test_case import LLMTestCase, SingleTurnParams
correctness_metric = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual_output is either an exact match to the "
        "expected_output, or a relevant statement that accurately conveys the "
        "same legal information."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    rubric=[
        Rubric(score_range=(0, 2), expected_outcome="Critical Failure/Incorrect — contradicts or misstates the law relative to the expected_output."),
        Rubric(score_range=(3, 4), expected_outcome="Poor/Significant Omissions — captures only a small fragment of the expected content or omits key legal elements."),
        Rubric(score_range=(5, 6), expected_outcome="Acceptable/Partially Complete — broadly correct but missing secondary details or nuance."),
        Rubric(score_range=(7, 9), expected_outcome="Good/Mostly Accurate — closely tracks the expected_output with only minor imprecision."),
        Rubric(score_range=(10, 10), expected_outcome="Excellent/Semantically Equivalent — fully and accurately conveys the same meaning as the expected_output, regardless of exact wording."),
    ],
    model = "gpt-5-mini-2025-08-07"
)

In [6]:
import math
import time
import pandas as pd
from tqdm import tqdm
from deepeval import evaluate
from deepeval.evaluate import DisplayConfig
from deepeval.evaluate import DisplayConfig, AsyncConfig

chunk_size = 100
max_concurrent = 20
max_retries = 3
base_delay = 2  # seconds, doubles each retry

test_cases = dataset.test_cases
n = len(test_cases)
num_chunks = math.ceil(n / chunk_size)

rows = []
failed_chunks = []

with tqdm(total=n, desc="Evaluating", unit="case") as pbar:
    for i in range(num_chunks):
        start = i * chunk_size
        chunk = test_cases[start:start + chunk_size]

        results = None
        for attempt in range(1, max_retries + 1):
            try:
                results = evaluate(
                    test_cases=chunk,
                    metrics=[correctness_metric],
                    display_config=DisplayConfig(show_indicator=True, print_results=False),
                    async_config=AsyncConfig(run_async=True, max_concurrent=max_concurrent),
                )
                break  # success, exit retry loop
            except Exception as e:
                if attempt == max_retries:
                    tqdm.write(f"Chunk {i} failed after {max_retries} attempts: {e}")
                    failed_chunks.append(i)
                else:
                    delay = base_delay * (2 ** (attempt - 1))
                    tqdm.write(f"Chunk {i} attempt {attempt} failed ({e}), retrying in {delay}s...")
                    time.sleep(delay)

        if results is not None:
            for test_result in results.test_results:
                if not test_result.metrics_data:
                    continue
                for metric in test_result.metrics_data:
                    rows.append({
                        "input": test_result.input,
                        "actual_output": test_result.actual_output,
                        "expected_output": test_result.expected_output,
                        "metric_name": metric.name,
                        "score": metric.score,
                        "reason": metric.reason,
                        "success": metric.success,
                    })

        pbar.update(len(chunk))

df = pd.DataFrame(rows)

Evaluating:   0%|          | 0/1435 [00:00<?, ?case/s]

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5-mini-2025-08-07, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=564417;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 138.37s | token cost: 0.30294424999999997 USD)
» Test Results (100 total tests):
   » Pass Rate: 49.0% | Passed: 49 | Failed: 51

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

...

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5-mini-2025-08-07, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=724672;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 87.06s | token cost: 0.10650724999999998 USD)
» Test Results (35 total tests):
   » Pass Rate: 45.71% | Passed: 16 | Failed: 19

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Evaluating: 100%|██████████| 1435/1435 [39:24<00:00,  1.65s/case]


In [7]:
# Save as Parquet
df.to_parquet("output/judgeRAG/correctness_results.parquet", engine="pyarrow", index=False)

In [8]:
import pandas as pd
eval = pd.read_parquet("output/judgeRAG/correctness_results.parquet")
eval

,input,actual_output,expected_output,metric_name,score,reason,success
0,Voor welke rechtbank moeten de ouders gaan in ...,Kort antwoord: geschillen over de vaststelling...,De familierechtbank behandelt alle kwesties in...,Correctness [GEval],0.6,The Actual Output correctly identifies the fam...,True
1,Kunnen de schuldeisers zich verzetten tegen de...,Kort: Uit de aangehaalde wetsartikelen blijkt ...,Als er gegronde redenen zijn om te twijfelen a...,Correctness [GEval],0.1,The Actual Output directly contradicts the Exp...,False
2,"Mijn buurman maakt veel lawaai, wat kan ik doen?",Kort antwoord: de stukken die u aanleverde heb...,Allereerst verdient dialoog de voorkeur! Luidr...,Correctness [GEval],0.3,The Actual Output responds to the question but...,False
3,Ik ben getrouwd. Moet ik een familiale aanspra...,Kort antwoord: Op basis van de aangeleverde we...,"U bent niet verplicht, maar het is aan te rade...",Correctness [GEval],0.6,"The actual output correctly states, based on t...",True
4,Ik heb een verklaring van wettelijke samenwoni...,Kort: de wet regelt voor wettelijk samenwonend...,Er is geen enkel verschil tussen een kind dat ...,Correctness [GEval],0.4,The Actual Output correctly identifies financi...,False
...,...,...,...,...,...,...,...
1430,Welke documenten en informatie mag het OCMW ei...,Kort gezegd mag het OCMW (het centrum) alle in...,U moet alle nuttige informatie meedelen zodat ...,Correctness [GEval],0.4,The actual output correctly states the core du...,False
1431,Eindigt de huurovereenkomst van korte duur aut...,Kort antwoord: Nee—het hangt af van de duur. I...,"Nee, een kortlopend huurcontract eindigt niet ...",Correctness [GEval],0.1,The response materially contradicts the expect...,False
1432,Ik vestig mijn domicilie. Ik woon in een carav...,Kort antwoord: u moet zich inschrijven in de g...,Als de caravan definitief is stilgelegd moet u...,Correctness [GEval],0.4,The actual output correctly states the general...,False
1433,Kan mijn verhuurder het kortlopende huurcontra...,"Kort gezegd: ja, maar onder strikte voorwaarde...",De eigenaar kan de huurovereenkomst van korte ...,Correctness [GEval],0.5,Scope: both outputs address whether a landlord...,True


In [9]:
#pip install ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35

In [10]:
from retrievers import DocsRetriever
def get_context(article_ids):
    article_tuples = [(id_val, 0) for id_val in article_ids]
    return DocsRetriever(article_tuples).documents()
outp["retrieved_contexts"] = outp["retrieved_ids"].apply(get_context)

In [11]:
outp

,query_id,query,answer,retrieved_ids,latency_sec,retrieved_contexts
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,Kort: in de verstrekte zoekresultaten is geen ...,"[27587, 2549, 27675, 14752, 20443, 13673, 9355...",52.305740,[Art. 36. § 1. Na advies van de in artikel 50 ...
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort antwoord: in principe oefenen de ouders h...,"[3790, 3791, 3794, 3812, 3797, 3811, 3796, 393...",17.306959,"[Art. 373. Wanneer de ouders samenleven, oefen..."
2,305,Moet ik branddetectors installeren in Brussel?,Ja. In het Brussels Gewest moet elke woning mi...,"[21608, 23173, 6930, 2247, 14630, 1783, 1704, ...",17.577047,[Art. D167bis. De personen die een boring of e...
3,1811,"Als ik mijn consumentenkrediet niet betaal, ku...",Kort: ja — in bepaalde omstandigheden kunnen (...,"[9505, 4856, 14614, 15632, 14533, 26606, 15646...",52.234624,[Art. 2.2.6.21. Rechtsmacht ten gronde § 1. In...
4,1558,Wat te doen als er een nieuwe huisgenoot in Br...,Kort en praktisch: geef de komst van de nieuwe...,"[4124, 4108, 26606, 152, 26740, 25728, 26739, ...",59.051880,[Art. 1728bis_BRUSSELS_HOOFDSTEDELIJK_GEWEST. ...
...,...,...,...,...,...,...
1430,1106,De bal van mijn kind is in de tuin van mijn bu...,Kort antwoord: ja — u mag uw bal laten terugge...,"[2715, 3780, 2782, 27419, 2710, 26462, 4278, 2...",30.579444,[Art. 3.67. Feitelijk gedogen van de eigenaar ...
1431,134,Hoe lang moet men wachten voordat men een besl...,U ontvangt in principe binnen 30 dagen na ontv...,"[26678, 26705, 26666, 26709, 26708, 26676, 266...",19.322466,[Art. 23.§ 1. De eerste betaling van het leefl...
1432,1031,Ik woon samen met mijn partner en/of mijn kind...,Kort antwoord: bij een verbreking van de samen...,"[3816, 106, 25835, 3791, 25833, 1074, 3724, 36...",24.015115,[Art. 387sexiesdecies. In deze titel worden ge...
1433,1715,Welke documenten en informatie mag het OCMW ei...,Kort gezegd mag het OCMW (het centrum) alle in...,"[26674, 26744, 26676, 27880, 26673, 1574, 2569...",82.663040,[Art. 19.§ 1. Met het oog op de toekenning van...


In [12]:
import os
import asyncio
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
from tqdm.asyncio import tqdm_asyncio

load_dotenv()

client = AsyncOpenAI(api_key=os.getenv("OPENAI_API"))
llm = llm_factory("gpt-4o-mini", client=client, max_tokens=16384)
scorer = Faithfulness(llm=llm)

CONCURRENCY = 15
TIMEOUT = 90
MAX_RETRIES = 5
BATCH_SIZE = 100

async def score_row(i, row, sem):
    question = row["query"]
    answer = row["answer"]
    context = row["retrieved_contexts"]
    score = None

    async with sem:
        for attempt in range(MAX_RETRIES):
            try:
                result = await asyncio.wait_for(
                    scorer.ascore(
                        user_input=question,
                        response=answer,
                        retrieved_contexts=context,
                    ),
                    timeout=TIMEOUT,
                )
                score = result.value
                break
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(min(2 ** attempt, 20))
                else:
                    print(f"Row {i}: failed after {MAX_RETRIES} attempts ({e})")

    return {"question": question, "answer": answer, "faithfulness_score": score}

async def run_evaluation(outp: pd.DataFrame) -> pd.DataFrame:
    sem = asyncio.Semaphore(CONCURRENCY)
    all_results = [] 

    num_batches = (len(outp) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(outp))
        batch_df = outp.iloc[start:end]

        tasks = [score_row(idx, row, sem) for idx, row in batch_df.iterrows()]
        batch_results = await tqdm_asyncio.gather(*tasks, desc=f"Batch {i + 1}/{num_batches}")
        all_results.extend(batch_results)

    results_df = pd.DataFrame(all_results)

    results_df.to_parquet("output/judgeRAG/faithfulness_results.parquet", index=False)
    return results_df

# Usage:
results_df = await run_evaluation(outp)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Batch 15/15: 100%|██████████| 35/35 [01:16<00:00,  2.19s/it]
